# 01 — Consolidar a RAIS (2017–2025): estabelecimentos de São José dos Campos

## Objetivo
Montar uma base **única e padronizada** de estabelecimentos da RAIS (microdados públicos do PDET/MTE) para **São José dos Campos (IBGE 354990)**, de **2017 a 2025**, pronta para o cruzamento posterior por CEP.

## Edição usada em cada ano
A escolha foi feita comparando os arquivos com o total nacional publicado no Sumário Executivo da RAIS 2025 (Tabela 18): **2023 = 11.768.420** e **2024 = 13.186.059** estabelecimentos. Só o `.COMT` bate nos dois anos. O `Parcial` não contém administração pública (zero estabelecimentos de natureza jurídica pública) e o `Legado` de 2023 não bate com o total publicado.

| Ano | Arquivo local (em `01_dados_brutos\rais\`) | Formato | Link no FTP (arquivo compactado) |
|---|---|---|---|
| 2017 | `2017\ESTB2017.txt` | antigo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2017/ESTB2017.7z` |
| 2018 | `2018\RAIS_ESTAB_PUB.txt` | antigo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2018/RAIS_ESTAB_PUB.7z` |
| 2019 | `2019\RAIS_ESTAB_PUB.txt` (pasta principal; o `Legado` é descartado) | antigo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2019/RAIS_ESTAB_PUB.7z` |
| 2020 | `2020\RAIS_ESTAB_PUB.txt` | antigo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2020/RAIS_ESTAB_PUB.7z` |
| 2021 | `2021\RAIS_ESTAB_PUB.txt` | antigo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2021/RAIS_ESTAB_PUB.7z` |
| 2022 | `2022\RAIS_ESTAB_PUB.txt.txt` | antigo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2022/RAIS_ESTAB_PUB.7z` |
| 2023 | `2023\RAIS_ESTAB_PUB.COMT` | novo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2023/RAIS_ESTAB_PUB.7z` |
| 2024 | `2024\RAIS_ESTAB_PUB.COMT` | novo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2024/RAIS_ESTAB_PUB.7z` |
| 2025 | `2025\RAIS_ESTAB_PUB.COMT` | novo | `ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/2025/RAIS_ESTAB_PUB.7z` |

## Os dois formatos de arquivo
| | Antigo (2017 a 2022) | Novo `.COMT` (2023 a 2025) |
|---|---|---|
| Separador | `;` | `,` |
| Colunas | 24 (`Tipo Estab` aparece duas vezes: código e texto) | 23 |
| Valores | com zeros e espaços de preenchimento | sem preenchimento, CNAE entre aspas |
| Posição do `CEP` (a partir de 0) | 23 | 22 |
| Posição do `IBGE Subsetor` | 22 | 21 |

As demais colunas usadas (posições 6 a 19) são **as mesmas nos dois formatos**. As colunas são lidas **por posição**, e o encoding é `latin-1`.

## Colunas da base final
| Coluna | Descrição |
|---|---|
| `ano` | Ano-base |
| `fonte_arquivo` | Link do arquivo no FTP do PDET |
| `municipio` | Código IBGE de 6 dígitos (sempre 354990) |
| `cep` | CEP em texto, 8 dígitos (zero à esquerda recuperado com `zfill(8)`) |
| `cep_valido` | `True` se o CEP tem 8 dígitos e é diferente de `00000000` |
| `cep_prefixo_fora_12` | `True` se o CEP válido **não** começa com `12`. Sinalização provisória; a validação definitiva será contra a base dos Correios e a base municipal de logradouros |
| `cnae20_classe` | 5 primeiros dígitos da subclasse (a coluna original muda de tamanho entre os formatos) |
| `cnae20_subclasse` | CNAE 2.0 subclasse, 7 dígitos |
| `ibge_subsetor` | Subsetor IBGE |
| `natureza_juridica` | Código de 4 dígitos |
| `tamanho_estab` | 1 = zero vínculos, 2 = até 4, 3 = 5 a 9, 4 = 10 a 19, 5 = 20 a 49, 6 = 50 a 99, 7 = 100 a 249, 8 = 250 a 499, 9 = 500 a 999, 10 = 1000 ou mais, -1 = ignorado |
| `tipo_estab` | Código numérico. Layout oficial: 1 = CNPJ, 3 = CEI, 9 = não identificado, -1 = ignorado. O código 6 aparece como `Cno` nos dados. O significado do 5 (formato novo) não está documentado no que temos |
| `ind_atividade_ano` | 1 = exerceu atividade no ano |
| `ind_cei_vinculado` | 1 = CEI vinculado |
| `ind_rais_negativa` | 1 = RAIS negativa (estabelecimento sem empregados) |
| `ind_simples` | 1 = optante pelo Simples |
| `qtd_vinc_clt`, `qtd_vinc_ativos`, `qtd_vinc_estatutarios` | Estoque de vínculos |

Fora da base, de propósito: `Ind Estab Participa PAT` (o código muda entre as eras), bairros SP/Fortaleza/RJ, distritos SP, região administrativa do DF e CNAE 95.

## Fluxo
| Etapa | O que faz | Células |
|---|---|---|
| 0 | Configuração, funções e pré-checagem dos arquivos | 1, 2 e 3 |
| 1 | Leitura em blocos e filtro do município (**lenta**, alguns minutos) | 4 |
| 2 | Tratamento e padronização (rápida, repetível) | 5 |
| 3 | Gravação dos `.parquet` e conferência | 6 |
| 4 | Resumo da série e alertas de variação | 7 |

## Como executar
- Rode as células **em ordem**. Se o kernel for reiniciado, rode de novo a Célula 1 e a Célula 2 antes de qualquer outra.
- A Célula 4 pula os anos que já estão em memória, então pode ser repetida após um erro sem reler tudo.
- Requisito: `pyarrow` (se faltar, rode `%pip install pyarrow` em uma célula).
- Saídas em `03_processamento\01_consolidar_rais\` (regeráveis: podem ser apagadas e recriadas).
- **Convenção do log:** texto terminando em `...` = etapa em andamento; texto terminando em `!` = etapa concluída.

## 0. Configuração

Define o log, as pastas, o **catálogo dos 9 anos** (arquivo local, formato e link do FTP), o mapa de posições das colunas por formato e os valores de referência usados na conferência cruzada.

**Valores de referência (conferência):** são os números medidos na etapa de diagnóstico (linhas e vínculos ativos de São José dos Campos em 2017, 2019, 2023, 2024 e 2025). Para 2018 e 2020 a 2022 não há referência, e a conferência aparece como "sem referência".

In [ ]:
"""Célula 1 - Preparação, pastas, catálogo dos anos e mapa de colunas."""
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "01_consolidar_rais"                # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import time                         # para medir quanto tempo cada etapa leva
import pandas as pd                 # biblioteca de tabelas (DataFrames)

# ---------------------------- Pastas do projeto ----------------------------
PASTA_BRUTOS = cfg.caminho("brutos.rais")          # 01_dados_brutos/rais: uma subpasta por ano, arquivos extraídos do FTP

# ---------------------------- Parâmetros gerais ----------------------------
COD_MUNICIPIO = cfg.projeto("municipio_ibge6")   # São José dos Campos (código IBGE de 6 dígitos)
PREFIXO_SJC = cfg.projeto("prefixo_cep")         # CEPs de SJC começam com 12 (teste provisório, frouxo)
TAMANHO_BLOCO = cfg.parametro("consolidacao.tamanho_bloco")   # linhas lidas por vez (controla o uso de memória)
LOG_A_CADA_N_BLOCOS = 5             # de quantos em quantos blocos o log de leitura aparece
LIMITE_VARIACAO_ALERTA = cfg.parametro("consolidacao.limite_variacao_alerta")   # alerta se o nº de estabelecimentos variar mais de 20% de um ano para o outro

# ---------------------------- Catálogo dos anos ----------------------------
FTP = "ftp://ftp.mtps.gov.br/pdet/microdados/RAIS"   # endereço-base do servidor do PDET

# Para cada ano: arquivo local, formato ("antigo" ou "novo") e link do arquivo compactado no FTP.
CATALOGO = {
    2017: {"arquivo": PASTA_BRUTOS / "2017" / "ESTB2017.txt",             "formato": "antigo", "url": f"{FTP}/2017/ESTB2017.7z"},
    2018: {"arquivo": PASTA_BRUTOS / "2018" / "RAIS_ESTAB_PUB.txt",       "formato": "antigo", "url": f"{FTP}/2018/RAIS_ESTAB_PUB.7z"},
    2019: {"arquivo": PASTA_BRUTOS / "2019" / "RAIS_ESTAB_PUB.txt",       "formato": "antigo", "url": f"{FTP}/2019/RAIS_ESTAB_PUB.7z"},   # pasta principal (Legado descartado)
    2020: {"arquivo": PASTA_BRUTOS / "2020" / "RAIS_ESTAB_PUB.txt",       "formato": "antigo", "url": f"{FTP}/2020/RAIS_ESTAB_PUB.7z"},
    2021: {"arquivo": PASTA_BRUTOS / "2021" / "RAIS_ESTAB_PUB.txt",       "formato": "antigo", "url": f"{FTP}/2021/RAIS_ESTAB_PUB.7z"},
    2022: {"arquivo": PASTA_BRUTOS / "2022" / "RAIS_ESTAB_PUB.txt.txt",   "formato": "antigo", "url": f"{FTP}/2022/RAIS_ESTAB_PUB.7z"},   # extensão duplicada é do arquivo original
    2023: {"arquivo": PASTA_BRUTOS / "2023" / "RAIS_ESTAB_PUB.COMT",      "formato": "novo",   "url": f"{FTP}/2023/RAIS_ESTAB_PUB.7z"},
    2024: {"arquivo": PASTA_BRUTOS / "2024" / "RAIS_ESTAB_PUB.COMT",      "formato": "novo",   "url": f"{FTP}/2024/RAIS_ESTAB_PUB.7z"},
    2025: {"arquivo": PASTA_BRUTOS / "2025" / "RAIS_ESTAB_PUB.COMT",      "formato": "novo",   "url": f"{FTP}/2025/RAIS_ESTAB_PUB.7z"},
}

# ---------------------------- Mapa de colunas ----------------------------
# Posição (contando a partir de 0) das colunas que valem nos DOIS formatos.
POS_COMUM = {
    "qtd_vinc_clt": 6, "qtd_vinc_ativos": 7, "qtd_vinc_estatutarios": 8,
    "ind_atividade_ano": 9, "ind_cei_vinculado": 10, "ind_rais_negativa": 12, "ind_simples": 13,
    "municipio": 14, "natureza_juridica": 15, "cnae20_subclasse": 17,
    "tamanho_estab": 18, "tipo_estab": 19,
}

# O que muda entre os formatos: o separador e a posição de IBGE Subsetor e CEP.
# (No formato antigo "Tipo Estab" aparece duas vezes, o que empurra as colunas seguintes 1 posição.)
FORMATOS = {
    "antigo": {"sep": ";", "posicoes": {**POS_COMUM, "ibge_subsetor": 22, "cep": 23}},
    "novo":   {"sep": ",", "posicoes": {**POS_COMUM, "ibge_subsetor": 21, "cep": 22}},
}

# Grupos de colunas, para converter os tipos de uma vez.
COLS_QTD = ["qtd_vinc_clt", "qtd_vinc_ativos", "qtd_vinc_estatutarios"]
COLS_IND = ["ind_atividade_ano", "ind_cei_vinculado", "ind_rais_negativa", "ind_simples"]

# Ordem final das colunas na base tratada.
ORDEM_COLUNAS = ["ano", "fonte_arquivo", "municipio", "cep", "cep_valido", "cep_prefixo_fora_12",
                 "cnae20_classe", "cnae20_subclasse", "ibge_subsetor", "natureza_juridica",
                 "tamanho_estab", "tipo_estab", *COLS_IND, *COLS_QTD]

# ---------------------------- Valores de referência (conferência cruzada) ----------------------------
# Medidos na etapa de diagnóstico, só para São José dos Campos.
REFERENCIA = {
    2017: {"linhas": 31_693, "vinculos_ativos": 187_441},
    2019: {"linhas": 31_688, "vinculos_ativos": 179_146},
    2023: {"linhas": 44_020, "vinculos_ativos": 213_282},
    2024: {"linhas": 48_371, "vinculos_ativos": 220_619},
    2025: {"linhas": 47_223, "vinculos_ativos": 228_391},
}
REF_FORA_PREFIXO_12 = {2017: 299, 2025: 5}   # CEPs que não começam com 12, medidos no diagnóstico de CEP

# Dicionários onde cada etapa guarda seu resultado (ficam na memória do notebook).
brutos = {}      # etapa 1: ano -> (tabela filtrada em texto, total de linhas do arquivo nacional)
tratados = {}    # etapa 2: ano -> tabela tratada

log.info("Configuração carregada!")

## 0.1 Funções

| Função | Papel |
|---|---|
| `fmt` | Formata números com ponto de milhar (leitura no log) |
| `ler_filtrar_municipio` | Lê o arquivo nacional **em blocos** e guarda só o município 354990 |
| `validar_etapa1` | Valida a leitura e o filtro |
| `tratar` | Padroniza textos, CEP, CNAE, tipos numéricos e acrescenta `ano` e `fonte_arquivo` |
| `validar_etapa2` | Valida o tratamento: perda, nulos, duplicados, CEP, coerência e conferência com a referência |
| `salvar_parquet` | Grava o `.parquet` de um ano, relê e confere |

**Decisões de leitura:**
- **Tudo é lido como texto** (`dtype=str`): preserva zeros à esquerda e evita conversões erradas.
- **Encoding `latin-1`:** confirmado pelo cabeçalho dos arquivos.
- **Leitura por posição** (`header=None`, `skiprows=1`), só das colunas necessárias.
- **Linhas duplicadas não são removidas:** não há ID do estabelecimento, então linhas idênticas são, muito provavelmente, estabelecimentos diferentes com os mesmos atributos.

In [ ]:
"""Célula 2 - Funções de leitura, tratamento e validação. Rode uma vez."""

def fmt(n) -> str:
    """Formata um número com ponto de milhar (padrão brasileiro). Ex.: 1234567 -> 1.234.567."""
    return f"{int(n):,}".replace(",", ".")


def ler_filtrar_municipio(ano: int):
    """Lê o arquivo bruto de um ano em blocos e mantém só as linhas do município alvo.

    Tudo é lido como texto (dtype=str) para não perder zeros à esquerda.
    Retorna (tabela filtrada, total de linhas lidas no arquivo nacional).
    """
    cfg = CATALOGO[ano]                                          # dados do ano (arquivo, formato, link)
    fmt_cfg = FORMATOS[cfg["formato"]]                           # separador e posições do formato desse ano
    pos_para_nome = {pos: nome for nome, pos in fmt_cfg["posicoes"].items()}   # ex.: {14: "municipio"}
    partes, total_lido = [], 0                                   # partes = pedaços filtrados; total_lido = contador

    leitor = pd.read_csv(
        cfg["arquivo"], sep=fmt_cfg["sep"],
        header=None, skiprows=1,           # lê por posição; pula a linha de cabeçalho
        usecols=sorted(pos_para_nome),     # só as colunas necessárias (economiza memória)
        dtype=str, keep_default_na=False,  # tudo como texto, sem transformar vazio em nulo
        encoding="latin-1",                # nunca dá erro de decodificação
        chunksize=TAMANHO_BLOCO,           # devolve um bloco por vez, em vez do arquivo inteiro
    )
    for i, bloco in enumerate(leitor, start=1):
        bloco = bloco.rename(columns=pos_para_nome)              # troca o número da posição pelo nome da coluna
        total_lido += len(bloco)                                 # soma as linhas deste bloco ao total
        eh_sjc = bloco["municipio"].str.strip() == COD_MUNICIPIO # True onde o município é São José dos Campos
        partes.append(bloco.loc[eh_sjc])                         # guarda só as linhas de SJC
        if i % LOG_A_CADA_N_BLOCOS == 0:                         # não escreve no log a cada bloco, para não poluir
            log.info(f"[{ano}] {fmt(total_lido)} linhas lidas...")
    return pd.concat(partes, ignore_index=True), total_lido     # junta os pedaços numa tabela só


def validar_etapa1(ano: int, df: pd.DataFrame, total_lido: int) -> None:
    """Validação 1: contagens da leitura e do filtro do município."""
    pct = len(df) / total_lido * 100 if total_lido else float("nan")
    log.info(f"[{ano}] VALIDAÇÃO 1 - linhas no arquivo nacional: {fmt(total_lido)}")
    log.info(f"[{ano}] VALIDAÇÃO 1 - linhas de {COD_MUNICIPIO}: {fmt(len(df))} ({pct:.2f}% do total)")
    if len(df) == 0:
        log.error(f"[{ano}] VALIDAÇÃO 1 - nenhuma linha do município! Confira o formato/posição da coluna.")
    log.info(f"[{ano}] VALIDAÇÃO 1 - municípios distintos após o filtro (esperado 1): {df['municipio'].str.strip().nunique()}")


def tratar(df_bruto: pd.DataFrame, ano: int) -> pd.DataFrame:
    """Padroniza textos, CEP (8 dígitos), CNAE classe, tipos numéricos; acrescenta ano e fonte_arquivo."""
    df = df_bruto.copy()                                         # trabalha numa cópia; o bruto fica intacto

    # 1) Tira espaços das pontas de todas as colunas e transforma texto vazio em nulo de verdade.
    for c in df.columns:
        df[c] = df[c].astype("string").str.strip().replace("", pd.NA)

    # 2) CEP: só dígitos e completa com zeros à esquerda até 8 dígitos (recupera o zero perdido).
    cep = df["cep"].str.replace(r"\D", "", regex=True)           # remove tudo que não for dígito
    log.info(f"[{ano}] CEPs com menos de 8 dígitos (zero à esquerda perdido): {fmt((cep.str.len() < 8).sum())}")
    df["cep"] = cep.str.zfill(8)                                 # ex.: 1415002 -> 01415002
    # CEP válido = 8 dígitos e diferente de 00000000.
    df["cep_valido"] = (df["cep"].str.len().eq(8) & df["cep"].ne("00000000")).fillna(False).astype(bool)

    # 3) Sinalização provisória: CEP válido que NÃO começa com 12 (fora da faixa de SJC e vizinhas).
    comeca_12 = df["cep"].str.startswith(PREFIXO_SJC).fillna(False).astype(bool)
    df["cep_prefixo_fora_12"] = df["cep_valido"] & ~comeca_12

    # 4) CNAE classe = 5 primeiros dígitos da subclasse (mesma regra para todos os anos).
    df["cnae20_classe"] = df["cnae20_subclasse"].str[:5]

    # 5) Colunas numéricas: texto -> inteiro. Valor que não converter vira nulo (e gera aviso).
    tipos = {**{c: "Int32" for c in COLS_QTD}, **{c: "Int8" for c in COLS_IND},
             "tamanho_estab": "Int16", "tipo_estab": "Int16"}
    for col, tipo in tipos.items():
        nulos_antes = int(df[col].isna().sum())
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(tipo)
        if int(df[col].isna().sum()) > nulos_antes:
            log.warning(f"[{ano}] {col}: valores não numéricos viraram nulo!")

    # 6) Colunas de identificação do ano e da origem dos dados.
    df["ano"] = pd.Series(ano, index=df.index, dtype="int16")
    df["fonte_arquivo"] = pd.Series(CATALOGO[ano]["url"], index=df.index, dtype="string")
    return df[ORDEM_COLUNAS]                                     # devolve na ordem final


def validar_etapa2(ano: int, df: pd.DataFrame, linhas_bruto: int) -> None:
    """Validação 2: perda, nulos, duplicados, CEP, coerência interna e conferência cruzada."""
    n = len(df)
    perda = (linhas_bruto - n) / linhas_bruto * 100 if linhas_bruto else float("nan")
    log.info(f"[{ano}] VALIDAÇÃO 2 - linhas: bruto={fmt(linhas_bruto)} | tratado={fmt(n)} | perda={perda:.2f}%")

    # Nulos por coluna (só mostra as que têm).
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    if nulos.empty:
        log.info(f"[{ano}] VALIDAÇÃO 2 - nulos: nenhum")
    for col, qtd in nulos.items():
        log.warning(f"[{ano}] VALIDAÇÃO 2 - nulos em {col}: {fmt(qtd)} ({qtd / n * 100:.2f}%)")

    # Duplicados exatos: só informativo (não há ID do estabelecimento; não são removidos).
    log.info(f"[{ano}] VALIDAÇÃO 2 - linhas totalmente duplicadas (informativo, não removidas): {fmt(df.duplicated().sum())}")

    # CEP.
    log.info(f"[{ano}] VALIDAÇÃO 2 - CEPs inválidos: {fmt((~df['cep_valido']).sum())} | CEPs distintos: {fmt(df['cep'].nunique())}")
    fora = int(df["cep_prefixo_fora_12"].sum())
    esperado_fora = REF_FORA_PREFIXO_12.get(ano)
    if esperado_fora is None:
        log.info(f"[{ano}] VALIDAÇÃO 2 - CEPs fora do prefixo {PREFIXO_SJC}: {fmt(fora)} ({fora / n * 100:.2f}%) | sem referência")
    else:
        status = "OK" if fora == esperado_fora else "DIVERGENTE"
        log.info(f"[{ano}] VALIDAÇÃO 2 - CEPs fora do prefixo {PREFIXO_SJC}: {fmt(fora)} ({fora / n * 100:.2f}%) | esperado {fmt(esperado_fora)} -> {status}")

    # Indicadores e coerência: rais negativa deveria ter zero vínculos ativos.
    negativa = int((df["ind_rais_negativa"] == 1).sum())
    com_vinculo = int((df["qtd_vinc_ativos"] > 0).sum())
    incoerentes = int(((df["ind_rais_negativa"] == 1) & (df["qtd_vinc_ativos"] > 0)).sum())
    log.info(f"[{ano}] VALIDAÇÃO 2 - rais negativa: {fmt(negativa)} | com vínculo ativo > 0: {fmt(com_vinculo)} | soma de vínculos ativos: {fmt(df['qtd_vinc_ativos'].sum())}")
    if incoerentes:
        log.warning(f"[{ano}] VALIDAÇÃO 2 - rais negativa com vínculos ativos > 0: {fmt(incoerentes)} (incoerente, investigar)")

    # Conferência cruzada com os valores medidos no diagnóstico.
    ref = REFERENCIA.get(ano)
    if ref is None:
        log.info(f"[{ano}] CONFERÊNCIA - sem referência para este ano")
    else:
        atual = {"linhas": n, "vinculos_ativos": int(df["qtd_vinc_ativos"].sum())}
        for chave, valor_ref in ref.items():
            status = "OK" if atual[chave] == valor_ref else "DIVERGENTE"
            log.info(f"[{ano}] CONFERÊNCIA - {chave}: obtido={fmt(atual[chave])} | referência={fmt(valor_ref)} -> {status}")


def salvar_parquet(df: pd.DataFrame, ano: int) -> Path:
    """Grava o parquet de um ano, relê o arquivo e confere linhas e tipo do CEP."""
    destino = PASTA_SAIDA / f"rais_estab_{ano}_sjc.parquet"
    df.to_parquet(destino, index=False)                          # grava sem a coluna de índice
    relido = pd.read_parquet(destino)                            # relê o que foi gravado
    status = "OK" if len(relido) == len(df) else "DIVERGENTE"
    log.info(f"[{ano}] VALIDAÇÃO 3 - linhas gravadas={fmt(len(df))} | relidas={fmt(len(relido))} -> {status}")
    log.info(f"[{ano}] VALIDAÇÃO 3 - cep dtype={relido['cep'].dtype} | exemplo={relido['cep'].iloc[0]} | colunas={len(relido.columns)}")
    return destino


log.info("Funções carregadas!")

## 0.2 Pré-checagem dos arquivos

Confere se os 9 arquivos existem nos caminhos do catálogo e mostra o tamanho de cada um. Se algum faltar, a célula para com erro **antes** de qualquer leitura demorada.

In [ ]:
"""Célula 3 - Pré-checagem: os 9 arquivos brutos existem?"""
log.info("Pré-checagem dos arquivos brutos...")
ausentes = []                                                    # anos cujo arquivo não foi encontrado
for ano, cfg in CATALOGO.items():
    existe = cfg["arquivo"].exists()                             # True se o arquivo está no caminho esperado
    tamanho_mb = cfg["arquivo"].stat().st_size / 1024**2 if existe else 0
    log.info(f"[{ano}] {'encontrado' if existe else 'AUSENTE'} | {tamanho_mb:.0f} MB | formato {cfg['formato']} | {cfg['arquivo']}")
    if not existe:
        ausentes.append(ano)
if ausentes:
    raise FileNotFoundError(f"Arquivos ausentes para os anos: {ausentes}. Confira o catálogo na Célula 1.")
log.info("Pré-checagem concluída!")

## 1. Etapa 1 — Leitura e filtro do município (9 anos)

Lê cada arquivo nacional em blocos de 500 mil linhas e guarda **apenas** as linhas do município 354990 em `brutos[ano]`. Cada ano leva de ~10 a ~20 segundos, então a célula toda leva alguns minutos.

**Validação 1 (no log), por ano:**
- total de linhas do arquivo nacional;
- linhas de São José dos Campos e % do total;
- municípios distintos após o filtro (**deve ser 1**; se for 0, o formato ou a posição da coluna está errado).

Anos já lidos são pulados, então a célula pode ser repetida após um erro sem recomeçar do zero.

In [ ]:
"""Célula 4 - Etapa 1: leitura e filtro do município, ano a ano."""
t_total = time.time()                                            # cronômetro da etapa inteira
for i, ano in enumerate(CATALOGO, start=1):                      # i = posição do ano na fila (1 a 9)
    if ano in brutos:                                            # já foi lido antes? então pula
        log.info(f"[{ano}] já está em memória, pulando ({i}/{len(CATALOGO)}).")
        continue
    t0 = time.time()
    log.info(f"[{ano}] Etapa 1 - Lendo e filtrando o arquivo em blocos ({i}/{len(CATALOGO)})...")
    brutos[ano] = ler_filtrar_municipio(ano)                     # guarda (tabela filtrada, total de linhas)
    validar_etapa1(ano, *brutos[ano])                            # contagens da validação 1
    log.info(f"[{ano}] Etapa 1 concluída em {time.time() - t0:.0f}s!")
log.info(f"Etapa 1 concluída para {len(brutos)} anos em {time.time() - t_total:.0f}s!")

## 2. Etapa 2 — Tratamento e padronização

| Campo | Regra |
|---|---|
| Todos os textos | Remove espaços das pontas; vazio vira nulo |
| `cep` | Só dígitos; completa com zeros à esquerda até 8 (`zfill(8)`) |
| `cep_valido` | 8 dígitos e diferente de `00000000`. **Nenhuma linha é descartada** |
| `cep_prefixo_fora_12` | CEP válido que não começa com `12`. Sinalização provisória |
| `cnae20_classe` | 5 primeiros dígitos da subclasse |
| `tamanho_estab`, `tipo_estab`, `ind_*`, `qtd_vinc_*` | Convertidos para inteiro; valor não numérico vira nulo e gera aviso |
| `ano`, `fonte_arquivo` | Ano-base e link do arquivo no FTP |

**Validação 2 (no log), por ano:** linhas bruto × tratado e taxa de perda (esperada **0,00%**), nulos por coluna, duplicados exatos (informativo, **não removidos**), CEPs inválidos e distintos, CEPs fora do prefixo 12, RAIS negativa × vínculos ativos (coerência), e **conferência cruzada** com os valores medidos no diagnóstico (2017, 2019, 2023, 2024 e 2025).

Esta etapa é rápida e pode ser repetida sem reler os arquivos brutos.

In [ ]:
"""Célula 5 - Etapa 2: tratamento e validação (rápida; pode ser repetida sem reler os arquivos)."""
t_total = time.time()
for ano, (df_bruto, _) in sorted(brutos.items()):                # um ano por vez, em ordem crescente
    log.info(f"[{ano}] Etapa 2 - Tratando e padronizando as colunas...")
    tratados[ano] = tratar(df_bruto, ano)                        # aplica as regras de limpeza
    validar_etapa2(ano, tratados[ano], linhas_bruto=len(df_bruto))
    log.info(f"[{ano}] Etapa 2 concluída!")
log.info(f"Etapa 2 concluída para {len(tratados)} anos em {time.time() - t_total:.0f}s!")

## 3. Etapa 3 — Gravação em parquet e conferência

Gera em `03_processamento/01_consolidar_rais/`:
- `rais_estab_<ano>_sjc.parquet`, um por ano (2017 a 2025);
- `rais_estab_sjc_consolidado.parquet`, com os 9 anos empilhados (coluna `ano`).

**Validação 3 (no log):** cada parquet é **relido** e comparado com a tabela em memória (linhas gravadas × relidas); confere também que o `cep` voltou como texto, com zeros à esquerda preservados. No consolidado, as linhas devem ser a soma dos anos.

In [ ]:
"""Célula 6 - Etapa 3: grava os parquets (um por ano + consolidado) e confere."""
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)                   # cria a pasta de saída se não existir

t_total = time.time()
for ano, df in sorted(tratados.items()):
    log.info(f"[{ano}] Etapa 3 - Gravando o parquet...")
    destino = salvar_parquet(df, ano)
    log.info(f"[{ano}] Etapa 3 concluída! {destino.name}")

log.info("Gerando o consolidado dos anos...")
consolidado = pd.concat(tratados.values(), ignore_index=True)    # empilha todos os anos
destino_cons = PASTA_SAIDA / "rais_estab_sjc_consolidado.parquet"
consolidado.to_parquet(destino_cons, index=False)
esperado = sum(len(d) for d in tratados.values())
status = "OK" if len(consolidado) == esperado else "DIVERGENTE"
log.info(f"VALIDAÇÃO 3 - consolidado: linhas={fmt(len(consolidado))} | esperado={fmt(esperado)} -> {status}")
log.info(f"VALIDAÇÃO 3 - anos no consolidado: {sorted(consolidado['ano'].unique().tolist())}")
log.info(f"Etapa 3 concluída em {time.time() - t_total:.0f}s! Consolidado: {destino_cons}")

## 4. Etapa 4 — Resumo da série

Lê o consolidado gravado e monta uma tabela por ano com:
- `estabelecimentos`: linhas da base;
- `com_vinculo_ativo`: estabelecimentos com `qtd_vinc_ativos > 0`;
- `rais_negativa`: estabelecimentos com `ind_rais_negativa = 1`;
- `vinculos_ativos`: soma do estoque de vínculos;
- `ceps_distintos` e `cep_fora_prefixo_12`;
- `var_estab_pct`: variação percentual de estabelecimentos em relação ao ano anterior.

**Alerta:** variações acima de 20% de um ano para o outro são marcadas com `WARNING`. O sumário da RAIS informa que, desde o ano-base 2023, os dados vêm exclusivamente do eSocial, o que sugere uma **possível quebra de série** entre 2019 e 2023. Esta tabela mostra se a mudança é gradual (2020 a 2022) ou abrupta. Ela **descreve** a série; a causa de cada variação não é testada aqui.

In [ ]:
"""Célula 7 - Etapa 4: resumo da série 2017-2025 e alertas de variação."""
log.info("Resumindo a série a partir do consolidado gravado...")
base = pd.read_parquet(PASTA_SAIDA / "rais_estab_sjc_consolidado.parquet")   # relê o arquivo gravado

faltam = sorted(set(CATALOGO) - set(base["ano"].unique().tolist()))          # anos do catálogo ausentes no consolidado
if faltam:
    log.error(f"VALIDAÇÃO 4 - anos ausentes no consolidado: {faltam}")
else:
    log.info("VALIDAÇÃO 4 - todos os anos do catálogo estão no consolidado!")

resumo = base.groupby("ano").agg(
    estabelecimentos=("ano", "size"),                                         # nº de linhas do ano
    com_vinculo_ativo=("qtd_vinc_ativos", lambda s: int((s > 0).sum())),     # estabelecimentos com vínculo ativo
    rais_negativa=("ind_rais_negativa", lambda s: int((s == 1).sum())),      # estabelecimentos sem empregados
    vinculos_ativos=("qtd_vinc_ativos", "sum"),                               # estoque total de vínculos
    ceps_distintos=("cep", "nunique"),
    cep_fora_prefixo_12=("cep_prefixo_fora_12", "sum"),
)
resumo["var_estab_pct"] = (resumo["estabelecimentos"].pct_change() * 100).round(1)   # variação sobre o ano anterior

for ano, linha in resumo.iterrows():
    var = linha["var_estab_pct"]
    texto = (f"[{ano}] estab={fmt(linha['estabelecimentos'])} | com vínculo={fmt(linha['com_vinculo_ativo'])} | "
             f"rais negativa={fmt(linha['rais_negativa'])} | vínculos ativos={fmt(linha['vinculos_ativos'])} | "
             f"CEPs distintos={fmt(linha['ceps_distintos'])}")
    if pd.notna(var) and abs(var) / 100 > LIMITE_VARIACAO_ALERTA:
        log.warning(f"{texto} | variação {var:+.1f}% (acima de {LIMITE_VARIACAO_ALERTA:.0%})")
    else:
        log.info(texto + (f" | variação {var:+.1f}%" if pd.notna(var) else ""))

display(resumo)                                                               # tabela formatada no Jupyter
log.info("Resumo da série concluído!")

## Checklist de conferência do log

- [ ] **Pré-checagem:** os 9 arquivos `encontrado`.
- [ ] **Validação 1:** municípios distintos = 1 em todos os anos; linhas de SJC plausíveis (dezenas de milhares).
- [ ] **Validação 2:** perda = 0,00% em todos os anos; `CONFERÊNCIA` em `OK` para 2017, 2019, 2023, 2024 e 2025; `CEPs fora do prefixo 12` em `OK` para 2017 e 2025.
- [ ] **Avisos (`WARNING`):** nulos em alguma coluna, valores não numéricos, RAIS negativa com vínculos ativos, variações acima de 20%.
- [ ] **Validação 3:** linhas gravadas = relidas; `cep` com dtype texto; consolidado = soma dos anos.
- [ ] **Validação 4:** os 9 anos presentes no consolidado.